In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import plotly.express as px
import plotly.graph_objects as go
import nbformat
import numpy as np


load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
database = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+mysqlconnector://{user}:{password}@{host}/{database}"
)

df_test = pd.read_sql(
    "SELECT COUNT(*) as total FROM fact_consumer_standing",
    con=engine
)

print(df_test)

   total
0    567


In [2]:
# --- QUERY 1: Impaired consumers over time ---

df_impaired = pd.read_sql("""
    SELECT d.period_label, d.year, d.month, f.value_millions
    FROM fact_consumer_standing f
    JOIN dim_period d ON f.period_id = d.period_id
    WHERE f.category = 'Impaired records (#)'
    ORDER BY d.year, d.month
""", con=engine)

print(df_impaired)

   period_label  year  month  value_millions
0        Jun'07  2007      6            6.11
1        Sep'07  2007      9            6.38
2        Dec'07  2007     12            6.45
3        Mar'08  2008      3            6.59
4        Jun'08  2008      6            6.79
..          ...   ...    ...             ...
66       Dec'23  2023     12            9.90
67       Mar'24  2024      3           10.09
68       Jun'24  2024      6           10.25
69       Sep'24  2024      9           10.19
70       Dec'24  2024     12           10.22

[71 rows x 4 columns]


In [3]:
# --- QUERY 2: Impaired consumers over time ---

df_rate = pd.read_sql("""
    SELECT 
        d.period_label, d.year, d.month,
        MAX(CASE WHEN f.category = 'Impaired records (#)' THEN f.value_millions END) as impaired,
        MAX(CASE WHEN f.category = 'Credit-active consumers (#)' THEN f.value_millions END) as credit_active,
        ROUND(
            MAX(CASE WHEN f.category = 'Impaired records (#)' THEN f.value_millions END) /
            MAX(CASE WHEN f.category = 'Credit-active consumers (#)' THEN f.value_millions END) * 100
        , 2) as impairment_rate_pct
    FROM fact_consumer_standing f
    JOIN dim_period d ON f.period_id = d.period_id
    GROUP BY d.period_id, d.period_label, d.year, d.month
    ORDER BY d.year, d.month
""", con=engine)


In [4]:
# --- QUERY 3: Enquiries by sector ---
df_enquiries = pd.read_sql("""
    SELECT d.period_label, d.year, d.month, f.category as sector, f.value as enquiries_millions
    FROM fact_enquries f
    JOIN dim_period d ON f.period_id = d.period_id
    WHERE f.source = 'enquiries_sector' AND f.category != 'Total'
    ORDER BY d.year, d.month, f.category
""", con=engine)

# --- QUERY 4: Good standing vs impaired ---
df_tug = pd.read_sql("""
    SELECT 
        d.period_label, d.year, d.month,
        MAX(CASE WHEN f.category = 'Good standing (#)' THEN f.value_millions END) as good_standing,
        MAX(CASE WHEN f.category = 'Impaired records (#)' THEN f.value_millions END) as impaired,
        MAX(CASE WHEN f.category = 'Credit-active consumers (#)' THEN f.value_millions END) as credit_active
    FROM fact_consumer_standing f
    JOIN dim_period d ON f.period_id = d.period_id
    GROUP BY d.period_id, d.period_label, d.year, d.month
    ORDER BY d.year, d.month
""", con=engine)

In [5]:

df_disputes = pd.read_sql("""
    SELECT 
        d.period_label, d.year, d.month,
        MAX(CASE WHEN f.category = 'Disputes lodged' THEN f.value END) as disputes_lodged,
        MAX(CASE WHEN f.category = 'Disputes resolved in favour of complainants' THEN f.value END) as resolved_favour,
        ROUND(
            MAX(CASE WHEN f.category = 'Disputes resolved in favour of complainants' THEN f.value END) /
            MAX(CASE WHEN f.category = 'Disputes lodged' THEN f.value END) * 100
        , 2) as resolution_rate_pct
    FROM fact_disputes_reports f
    JOIN dim_period d ON f.period_id = d.period_id
    WHERE f.source = 'disputes'
    GROUP BY d.period_id, d.period_label, d.year, d.month
    ORDER BY d.year, d.month
""", con=engine)

In [6]:
for df in [df_impaired, df_rate, df_tug, df_disputes]:
    df.sort_values(['year', 'month'], inplace=True)
    df.reset_index(drop=True, inplace=True)

In [7]:
fig1 = px.line(
    df_impaired,
    x='period_label', y='value_millions',
    title='Impaired Consumers in South Africa (2007–2024)',
    labels={'value_millions': 'Consumers (Millions)', 'period_label': 'Period'},
    template='plotly_dark'
)
covid_idx = df_impaired[df_impaired['period_label'].str.strip() == "Mar'20"].index[0]
fig1.update_traces(line_color='#e74c3c', line_width=2)
fig1.add_vline(x=covid_idx, line_dash='dash', line_color='white', annotation_text='COVID-19',annotation_font_color='white')
fig1.show(renderer="browser")

In [8]:
fig2 = px.area(
    df_rate,
    x='period_label', y='impairment_rate_pct',
    title='Impairment Rate — % of Credit-Active Consumers (2007–2024)',
    labels={'impairment_rate_pct': 'Impairment Rate (%)', 'period_label': 'Period'},
    template='plotly_dark'
)
fig2.update_traces(line_color='#e67e22', fillcolor='rgba(230,126,34,0.2)')
fig2.show(renderer='browser')

# --- CHART 3: Enquiries by sector over time ---
fig3 = px.line(
    df_enquiries,
    x='period_label', y='enquiries_millions',
    color='sector',
    title='Credit Enquiries by Sector (2008–2024)',
    labels={'enquiries_millions': 'Enquiries (Millions)', 'period_label': 'Period'},
    template='plotly_dark'
)
fig3.show(renderer='browser')

In [9]:
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=df_tug['period_label'], y=df_tug['good_standing'],
    name='Good Standing', line=dict(color='#2ecc71', width=2)
))
fig4.add_trace(go.Scatter(
    x=df_tug['period_label'], y=df_tug['impaired'],
    name='Impaired', line=dict(color='#e74c3c', width=2)
))
fig4.update_layout(
    title='Good Standing vs Impaired Consumers (2007–2024)',
    xaxis_title='Period', yaxis_title='Consumers (Millions)',
    template='plotly_dark'
)
fig4.show(renderer='browser')


In [10]:
fig5 = go.Figure()
fig5.add_trace(go.Bar(
    x=df_disputes['period_label'], y=df_disputes['disputes_lodged'],
    name='Disputes Lodged', marker_color='#3498db'
))
fig5.add_trace(go.Scatter(
    x=df_disputes['period_label'], y=df_disputes['resolution_rate_pct'],
    name='Resolution Rate %', yaxis='y2',
    line=dict(color='#f1c40f', width=2)
))
fig5.update_layout(
    title='Disputes Lodged & Resolution Rate (2007–2024)',
    xaxis_title='Period',
    yaxis=dict(title='Disputes Lodged'),
    yaxis2=dict(title='Resolution Rate (%)', overlaying='y', side='right'),
    template='plotly_dark',
    barmode='group'
)
fig5.show(renderer='browser')



In [11]:
df = pd.read_sql("""
    SELECT d.period_label, d.year, d.month, f.value_millions
    FROM fact_consumer_standing f
    JOIN dim_period d ON f.period_id = d.period_id
    WHERE f.category = 'Impaired records (#)'
    ORDER BY d.year, d.month
""", con=engine)

In [12]:
df['qoq_change'] = df['value_millions'].diff()
df['direction'] = np.where(df['qoq_change'] < 0, 'recovery', 
                  np.where(df['qoq_change'] > 0, 'deterioration', 'flat'))

# --- STEP 2: IDENTIFY RECOVERY EPISODES ---
# A recovery episode = consecutive quarters where direction == 'recovery'
episodes = []
in_episode = False
episode_start = None
episode_length = 0
episode_start_value = None

for i, row in df.iterrows():
    if row['direction'] == 'recovery':
        if not in_episode:
            in_episode = True
            episode_start = row['period_label']
            episode_start_idx = i
            episode_length = 1
            episode_start_value = df.loc[i-1, 'value_millions'] if i > 0 else row['value_millions']
        else:
            episode_length += 1
    else:
        if in_episode:
            episode_end_value = df.loc[i-1, 'value_millions']
            total_reduction = episode_start_value - episode_end_value
            episodes.append({
                'start_period': episode_start,
                'end_period': df.loc[i-1, 'period_label'],
                'duration_quarters': episode_length,
                'start_value': round(episode_start_value, 2),
                'end_value': round(episode_end_value, 2),
                'total_reduction_millions': round(total_reduction, 2),
                'sustained': episode_length >= 3
            })
            in_episode = False
            episode_length = 0

if in_episode:
    episode_end_value = df['value_millions'].iloc[-1]
    total_reduction = episode_start_value - episode_end_value
    episodes.append({
        'start_period': episode_start,
        'end_period': df['period_label'].iloc[-1],
        'duration_quarters': episode_length,
        'start_value': round(episode_start_value, 2),
        'end_value': round(episode_end_value, 2),
        'total_reduction_millions': round(total_reduction, 2),
        'sustained': episode_length >= 3
    })

df_episodes = pd.DataFrame(episodes)

print("=== RECOVERY EPISODES ===")
print(df_episodes.to_string(index=False))
print(f"\nTotal recovery episodes: {len(df_episodes)}")
print(f"Sustained recoveries (3+ quarters): {df_episodes['sustained'].sum()}")
print(f"Average recovery duration: {df_episodes['duration_quarters'].mean():.1f} quarters")
print(f"Longest recovery: {df_episodes['duration_quarters'].max()} quarters")
print(f"Average reduction per episode: {df_episodes['total_reduction_millions'].mean():.2f}M consumers")

# --- STEP 3: VISUALISE ---
DARK_BG = '#0a0e1a'
CARD_BG = '#111827'
ACCENT_RED = '#ef4444'
ACCENT_GREEN = '#10b981'
ACCENT_AMBER = '#f59e0b'
TEXT_COLOR = '#e2e8f0'
GRID_COLOR = '#1e2a3a'

fig = go.Figure()

# Base impairment line
fig.add_trace(go.Scatter(
    x=df['period_label'],
    y=df['value_millions'],
    mode='lines',
    name='Impaired Consumers',
    line=dict(color=ACCENT_RED, width=2),
    fill='tozeroy',
    fillcolor='rgba(239,68,68,0.05)'
))

# Highlight recovery episodes as green shaded regions
for _, ep in df_episodes.iterrows():
    start_idx = df[df['period_label'] == ep['start_period']].index[0]
    end_idx = df[df['period_label'] == ep['end_period']].index[0]
    
    fig.add_vrect(
        x0=start_idx, x1=end_idx,
        fillcolor='rgba(16,185,129,0.1)',
        layer='below',
        line_width=0,
        annotation_text='↓' if ep['sustained'] else '',
        annotation_position='top left',
        annotation_font_color=ACCENT_GREEN,
        annotation_font_size=14
    )

# Mark sustained recoveries with a distinct marker
sustained = df_episodes[df_episodes['sustained'] == True]
for _, ep in sustained.iterrows():
    mid_idx = df[df['period_label'] == ep['start_period']].index[0]
    fig.add_annotation(
        x=mid_idx,
        y=df.loc[mid_idx, 'value_millions'],
        text=f"Sustained<br>{ep['duration_quarters']}Q",
        showarrow=True,
        arrowhead=2,
        arrowcolor=ACCENT_GREEN,
        font=dict(color=ACCENT_GREEN, size=10),
        bgcolor='rgba(16,185,129,0.1)',
        bordercolor=ACCENT_GREEN,
        borderwidth=1
    )

fig.update_layout(
    title=dict(
        text='The Invisible Recovery — Every Green Zone Was Temporary',
        font=dict(size=16, color=TEXT_COLOR)
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(color=TEXT_COLOR, family='DM Sans, sans-serif'),
    xaxis=dict(
        gridcolor=GRID_COLOR, showgrid=True,
        tickangle=-45, tickfont=dict(size=9)
    ),
    yaxis=dict(
        gridcolor=GRID_COLOR, showgrid=True,
        title='Impaired Consumers (Millions)'
    ),
    margin=dict(l=40, r=20, t=60, b=60),
    hovermode='x unified',
    showlegend=True
)

fig.show(renderer='browser')

fig2 = go.Figure(data=[go.Table(
    header=dict(
        values=['Start Period', 'End Period', 'Duration (Quarters)',
                'Reduction (Millions)', 'Sustained (3Q+)'],
        fill_color='#1e2a3a',
        font=dict(color=TEXT_COLOR, size=12),
        align='left',
        height=36
    ),
    cells=dict(
        values=[
            df_episodes['start_period'],
            df_episodes['end_period'],
            df_episodes['duration_quarters'],
            df_episodes['total_reduction_millions'],
            df_episodes['sustained'].map({True: '✅ Yes', False: '❌ No'})
        ],
        fill_color=[['#111827' if i % 2 == 0 else '#0f172a'
                     for i in range(len(df_episodes))]],
        font=dict(color=TEXT_COLOR, size=11),
        align='left',
        height=32
    )
)])

fig2.update_layout(
    title=dict(text='Recovery Episodes Summary', font=dict(size=14, color=TEXT_COLOR)),
    paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=0, r=0, t=40, b=0)
)

fig2.show(renderer='browser')


total = len(df_episodes)
sustained_count = int(df_episodes['sustained'].sum())
avg_duration = df_episodes['duration_quarters'].mean()
max_reduction = df_episodes['total_reduction_millions'].max()

print("\n=== THE VERDICT ===")
print(f"Across 17 years and 71 quarters:")
print(f"- {total} recovery episodes identified")
print(f"- Only {sustained_count} lasted 3 or more consecutive quarters")
print(f"- Average recovery lasted just {avg_duration:.1f} quarters before reversing")
print(f"- The largest single reduction was {max_reduction:.2f}M consumers")
print(f"- Conclusion: SA consumer credit has NEVER achieved a sustained recovery")

=== RECOVERY EPISODES ===
start_period end_period  duration_quarters  start_value  end_value  total_reduction_millions  sustained
      Sep'10     Sep'10                  1         8.59       8.49                      0.10      False
      Mar'14     Mar'14                  1         9.93       9.60                      0.33      False
      Sep'15     Mar'16                  3        10.53       9.55                      0.98       True
      Dec'16     Mar'17                  2         9.85       9.69                      0.16      False
      Dec'17     Sep'18                  4         9.87       8.98                      0.89       True
      Mar'19     Mar'19                  1        10.16      10.15                      0.01      False
      Dec'19     Jun'20                  3        10.80      10.00                      0.80       True
      Dec'20     Jun'21                  3        10.64      10.07                      0.57       True
      Dec'21     Dec'21               

In [13]:
df_impaired.to_csv('../data/df_impaired.csv', index=False)
df_rate.to_csv('../data/df_rate.csv', index=False)
df_enquiries.to_csv('../data/df_enquiries.csv', index=False)
df_tug.to_csv('../data/df_tug.csv', index=False)
df_disputes.to_csv('../data/df_disputes.csv', index=False)
df_episodes.to_csv('../data/df_recovery.csv', index=False)